# 从 CSV 更新结果 README

发布规格为十个主实验表、四幅全量图、一张消融表和两幅消融图。每次评分写回 CSV 后执行本 Notebook，报告会重新读取数值并标记各表最优。两个消融图分别是 UMAP 和包含整体及分类型面板的边缘对齐图。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

def update_results_readme(root, figure_prefix=None):
    root = Path(root).resolve()
    if figure_prefix is None:
        figure_prefix = 'figures/'
    columns = ['A1 1-WD↓','A1 MSE↓','A1 Pearson↑','A2 Pearson↑','A3 miLISI↑']
    new_methods = {'PeakVI','DiTSim'}
    cases = ['liu2023 全数据集','liu2023 GSE213221','liu2023 GSE215213','liu2023 GSE215342',
             'liu2021 GSE130553','liu2021 GSE131354','liu2021 GSE131360',
             'tian2023 GSE167119','tian2023 GSE167577','tian2023 GSE167978']
    tables = [(case,pd.read_csv(root / '逐数据集表' / f'{case}.csv',float_precision='round_trip')) for case in cases]
    finished = sum(table['方法'].isin(new_methods).sum() for _, table in tables)
    filtered = (root / 'Subclass过滤汇总.json').is_file()
    scored = sum(len(table) for _, table in tables)
    lines = ['# 主实验与消融结果','',
        '**交付清单**：10 张主实验表、4 幅全量图、1 张消融表和 2 幅消融图。','',
        '每次评分更新后，用「更新结果README.ipynb」从 CSV 刷新本页。加粗值为该表最优；N/A 表示该指标无法定义。','',
        f'新增两种基线已评分 {finished}/{len(cases) * len(new_methods)} 个方法与数据集组合。以下表格列出已完成评分的结果。','',
        '新基线使用真实训练集的逐区域中位数填补缺失，生成后裁剪到 [0,1]。各方法生成细胞数和逐 Subclass 配额与测试集相同。','']
    if filtered:
        lines[6:8] = [
            f'过滤后主实验已评分 {scored}/{len(cases) * (4 + len(new_methods))} 个方法与数据集组合，其中新增基线 {finished}/{len(cases) * len(new_methods)}。',
            '',
            '十组均按真实训练集的 Subclass 细胞数过滤。训练少于 10 个细胞的类别从本组训练和测试中同步移除。'
            '原训练／测试归属、区域轴和 1,000 对复制区域沿用原设定。[过滤记录](Subclass过滤汇总.json)',
            '',
            '本页的 simscDNAm 主实验和消融使用 50 步 DDIM。复跑时在训练配置中设定 `DDIM_SAMPLING_STEPS=50`。',
            '',
        ]
    def append_table(name, table):
        assert table.columns.tolist() == ['方法',*columns], name
        assert table['方法'].is_unique, name
        lines.extend([f'**{name}**','', '| 方法 | ' + ' | '.join(columns) + ' |',
            '| :--- | ---: | ---: | ---: | ---: | ---: |'])
        for _, row in table.iterrows():
            values=[str(row['方法'])]
            for column in columns:
                value=row[column]
                best=table[column].min() if column in columns[:2] else table[column].max()
                shown='N/A' if not np.isfinite(value) else f'{value:.3e}' if column==columns[1] else f'{value:.5f}'
                values.append(f'**{shown}**' if np.isfinite(value) and np.isclose(value,best,rtol=0,atol=1e-12) else shown)
            lines.append('| '+' | '.join(values)+' |')
        lines.append('')
    def append_figure(title, stem):
        available = all((root/'figures'/f'{stem}.{extension}').is_file() for extension in ('png','pdf'))
        if filtered and not available:
            lines.extend([f'**{title}**','', '图待生成。',''])
            return
        assert available, stem
        lines.extend([f'**{title}**','',f'![{title}]({figure_prefix}{stem}.png)',''])
    for name, table in tables:
        append_table(name, table)
        if name == 'liu2021 GSE131354' and table['方法'].eq('simscDNAm（探索性）').any():
            lines.extend(['该行来自内部 miLISI 最高的候选 frozen_beta_1e6。'
                          '四个候选均未达到原内部重构与多样性验收门槛。',''])
    lines.extend(['Liu2021 和 Tian2023 的六个子集使用逐元素核验后恢复的真实矩阵，并按本轮划分重新训练和评分。','',
        '边缘图采用 [Liu 等，Nature 2023 图 1d](https://www.nature.com/articles/s41586-023-06805-y/figures/1) 的 Tle4 展示范围。各窗口宽 10 kb，n 表示该区域的有限观测数。',''])
    for title, stem in [
        ('全量：多区域边缘分布','A1_marginal_regions'),
        ('全量：单区域按 Subclass 的边缘分布','A1_region_by_type'),
        ('全量：复制区域 Pearson 小提琴图','A2_duplicate_Pearson'),
        ('全量：真实与合成数据联合 UMAP','A3_joint_UMAP'),
    ]:
        append_figure(title, stem)
    ablation = pd.read_csv(root/'tables/liu2023全量消融.csv',float_precision='round_trip')
    append_table('Liu2023 全量后处理消融', ablation)
    variants = '原始生成、仅训练缺失掩码和完整模型' if filtered else '原始生成、仅掩码、仅对齐和完整模型'
    lines.extend([f'消融固定同一训练模型和采样种子，对比{variants}。UMAP 各面板独立拟合，miLISI 来自共享的训练参照 PCA 空间。',''])
    append_figure('消融：UMAP','Ablation_A3_joint_UMAP')
    append_figure('消融：边缘对齐，上为全部类型合并，下为按 Subclass','Ablation_marginal_alignment')
    rendered='\n'.join(lines)
    assert rendered.count('| 方法 | A1 1-WD↓') == 11
    if not filtered or scored == len(cases) * (4 + len(new_methods)) and len(ablation) == 3:
        assert rendered.count('![') == 6
    for name in ('README.md','逐数据集结果.md'):
        target = root/name
        partial = target.with_suffix('.md.partial')
        partial.write_text(rendered,encoding='utf-8')
        partial.replace(target)
    return int(finished)


In [ ]:
from data.paths import OUTPUTS, SERVER
root = Path(os.environ.get('RESULTS_ROOT', OUTPUTS))
prefix = os.environ.get('RESULTS_FIGURE_PREFIX')
if prefix is None and SERVER:
    relative = root.resolve().relative_to(OUTPUTS.resolve())
    prefix = '/files/data/xb/outputs/simscDNAm/' + (relative / 'figures').as_posix() + '/'
print('README 已更新，已评分新增组合数', update_results_readme(root, prefix))